# Graded Response Model — TMA (Single Scale)

Fits a single-dimensional GRM to all 50 TMA items. With binary responses (K=2), this is equivalent to a 2PL IRT model.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

## 1. Load Data

In [ ]:
from bayesianquilts.data.tma import get_data, item_keys, response_cardinality

df, num_people = get_data(polars_out=True)
print(f"Dataset shape: {df.shape}")
print(f"Number of people: {num_people}")
print(f"Number of items: {len(item_keys)}")
print(f"Response cardinality: {response_cardinality} (binary: 0=False, 1=True)")
df.head()

In [ ]:
SUBSAMPLE_N = num_people
sub_df = df
print(f"Using full dataset: N = {SUBSAMPLE_N}")

## 2. Prepare Data

In [ ]:
def make_data_dict(dataframe):
    data = {}
    for col in dataframe.columns:
        arr = dataframe[col].to_numpy().astype(np.float64)
        data[col] = arr
    data['person'] = np.arange(len(dataframe), dtype=np.float64)
    return data

batch = make_data_dict(sub_df)

# Check for missing/invalid values
n_bad = sum(
    np.sum(np.isnan(batch[k]) | (batch[k] < 0) | (batch[k] >= response_cardinality))
    for k in item_keys
)
print(f"Bad/missing values: {n_bad}")

BATCH_SIZE = 256
steps_per_epoch = int(np.ceil(SUBSAMPLE_N / BATCH_SIZE))
print(f"N: {SUBSAMPLE_N}, Batch size: {BATCH_SIZE}, Steps per epoch: {steps_per_epoch}")

def data_factory():
    indices = np.arange(SUBSAMPLE_N)
    np.random.shuffle(indices)
    for start in range(0, SUBSAMPLE_N, BATCH_SIZE):
        end = min(start + BATCH_SIZE, SUBSAMPLE_N)
        idx_batch = indices[start:end]
        yield {k: v[idx_batch] for k, v in batch.items()}

## 3. Fit Baseline GRM (Ignorable Missingness)

In [ ]:
from bayesianquilts.irt.grm import GRModel

model_baseline = GRModel(
    item_keys=item_keys,
    num_people=SUBSAMPLE_N,
    dim=1,
    kappa_scale=0.1,
    response_cardinality=response_cardinality,
    dtype=jnp.float64,
)

NUM_EPOCHS = 200

losses_baseline, params_baseline = model_baseline.fit(
    data_factory,
    batch_size=BATCH_SIZE,
    dataset_size=SUBSAMPLE_N,
    num_epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    learning_rate=2e-4,
    patience=10,
    zero_nan_grads=True,
)

print(f"Baseline final loss: {losses_baseline[-1]:.2f}")

In [ ]:
model_baseline.save_to_disk('grm_baseline')

## 4. Fit MICEBayesianLOO Imputation Model

In [ ]:
from bayesianquilts.imputation.mice_loo import MICEBayesianLOO

# Convert to pandas with NaN for missing values (-1 marks missing in TMA)
imputation_df = sub_df.select(item_keys).to_pandas()
imputation_df = imputation_df.replace(-1, np.nan)
print(f"Imputation DataFrame shape: {imputation_df.shape}")
print(f"NaN count: {imputation_df.isna().sum().sum()}")

mice_loo = MICEBayesianLOO(
    random_state=42,
    prior_scale=1.0,
    pathfinder_num_samples=100,
    pathfinder_maxiter=50,
    batch_size=512,
    verbose=True,
)

mice_loo.fit_loo_models(
    X_df=imputation_df,
    n_top_features=50,
    n_jobs=-1,
    fit_zero_predictors=True,
    seed=42,
)

print(f"\nFitted variable names: {mice_loo.variable_names[:5]}...")
print(f"Zero-predictor models: {len(mice_loo.zero_predictor_results)}")
print(f"Univariate models: {len(mice_loo.univariate_results)}")

In [ ]:
mice_loo.save('mice_loo_model.yaml')

## 5. Fit GRM with Analytic Rao-Blackwellized Imputation

In [ ]:
model_imputed = GRModel(
    item_keys=item_keys,
    num_people=SUBSAMPLE_N,
    dim=1,
    kappa_scale=0.1,
    response_cardinality=response_cardinality,
    dtype=jnp.float64,
    imputation_model=mice_loo,
)

model_imputed.validate_imputation_model()
print("Imputation model validation passed.")

losses_imputed, params_imputed = model_imputed.fit(
    data_factory,
    batch_size=BATCH_SIZE,
    dataset_size=SUBSAMPLE_N,
    num_epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    learning_rate=2e-4,
    patience=10,
    zero_nan_grads=True,
)

print(f"Imputed final loss: {losses_imputed[-1]:.2f}")

In [ ]:
model_imputed.save_to_disk('grm_imputed')

## 6. Compare Results

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses_baseline, label='Baseline (ignorable)', alpha=0.8)
plt.plot(losses_imputed, label='Analytic Rao-Blackwell imputation', alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Loss (neg ELBO)')
plt.title('Training Loss: Baseline vs Analytic Imputation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def calibrate_manually(model, n_samples=32, seed=42):
    surrogate = model.surrogate_distribution_generator(model.params)
    key = jax.random.PRNGKey(seed)
    samples = surrogate.sample(n_samples, seed=key)
    expectations = {k: jnp.mean(v, axis=0) for k, v in samples.items()}
    model.calibrated_expectations = expectations
    model.surrogate_sample = samples

calibrate_manually(model_baseline, n_samples=32, seed=101)
calibrate_manually(model_imputed, n_samples=32, seed=102)

In [ ]:
disc_base = np.array(model_baseline.calibrated_expectations['discriminations']).flatten()
disc_imp = np.array(model_imputed.calibrated_expectations['discriminations']).flatten()

fig, ax = plt.subplots(figsize=(10, 12))
y_pos = np.arange(len(item_keys))
width = 0.35
ax.barh(y_pos - width/2, disc_base, width, label='Baseline', alpha=0.7, edgecolor='black')
ax.barh(y_pos + width/2, disc_imp, width, label='Imputed', alpha=0.7, edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(item_keys)
ax.set_xlabel('Discrimination')
ax.set_title('Item Discriminations: Baseline vs Imputed')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
ab_base = np.array(model_baseline.calibrated_expectations['abilities']).flatten()
ab_imp = np.array(model_imputed.calibrated_expectations['abilities']).flatten()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ab_base, bins=30, alpha=0.5, label='Baseline', edgecolor='black')
ax.hist(ab_imp, bins=30, alpha=0.5, label='Imputed', edgecolor='black')
ax.set_xlabel('Ability (TMA latent trait)')
ax.set_ylabel('Count')
ax.set_title('Ability Distribution: Baseline vs Imputed')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated fitting a single-dimensional Graded Response Model to the
Taylor Manifest Anxiety Scale (TMA) dataset (50 binary items, K=2). With binary responses,
the GRM reduces to a 2PL IRT model.

Two models were fitted:

1. **Baseline (ignorable missingness)**: Missing responses have their log-likelihood
   contributions zeroed out, assuming the missingness mechanism is ignorable.
2. **Analytic Rao-Blackwellized imputation**: A `MICEBayesianLOO` imputation model
   provides categorical PMFs for missing cells, which are used to analytically
   marginalize over the imputation distribution during model fitting.

The discrimination and ability estimates from both approaches can be compared to assess
the impact of the missingness handling strategy on parameter recovery.